Phenyo Thato Molete 216038155 
Assignment 4: Wallet & Digital Signature Lab 
BLCH9X2

Blockchain wallet implementation: 
Wallet: a class which wraps a pair of ECDSA keys using the SECP256k1 curve (this is the same type curve used by Bitcoin and Ethreum)
Adderess derivation: Converts the public key into the wallet's blockchain address
Signing: Uses the waller owner's private key to create a digital signature for the transaction
Verification: The public key is used to check that the signature is valid and that the transaction has not been altered. 
canonical_tax_payload(): creates the exact deterministic bytes which represent a transaction that should be signed.


In [1]:
from __future__ import annotations

In [2]:
import hashlib
import json
from typing import Optional

In [3]:
!pip install ecdsa

In [4]:
from ecdsa import BadSignatureError,SECP256k1, SigningKey, VerifyingKey

To derive a simplified wallet address from a shortened public key, first the public key is represented as raw bytes using #def pubkey_to_address(pubkey_bytes: bytes) 
Then the function returns a string from # -> str:
The public key is then hashed using SHA-256 which takes the public key and produces a 256-bit (32-byte) hash.
The SHA-256 hash, which is usually 32 bytes hash is converted to a hexa decimal using #hexdigest() which represents those bytes as 64 hexadecimal characters
Finally the 64 character hash is truncated to 40 hexadecimal characters using [:40]

In [5]:
def pubkey_to_address(pubkey_bytes: bytes) -> str:
    return hashlib.sha256(pubkey_bytes).hexdigest()[:40]

canonical_tx_payload() is the agreement between the signing and verification sides about exactly what was signed.
It takes the four transaction fields (the sender, recipient, amount and the time the transaction was created) and returns bytes, since the ecdsa signing function ultimately signs bytes.
The transaction is information is first turned into a dictionary, and subsequently uses #sort_keys=True to force the keys into alphebetical order to ensure uniform representation. 
The separators are an additional form of cannonicalisation which removes unnecessary spaces with the ultimate goal of uniformity to ensure signature verification works. 
Finally, encode("utf-8") converts the JSON string into bytes and these bytes are what the wallet signs. 
The signature field is prposefully excluded.

In [6]:
def canonical_tx_payload(sender: str, recipient: str, amount: float, timestamp: float) ->bytes:
    body = { 
        "amount": amount, 
        "recipient": recipient, 
        "sender": sender, 
        "timestamp": timestamp,
    }
    return json.dumps(body, sort_keys=True, separators=(",", ":")).encode("utf-8")

The main wallet class is now used, bringing together the above applied functions, publickey_toaddress and canonical_tx_payload()
First, the wallet is created. The wallet stores _sk->the private key and _vk->the public key. The private key is secret and the public key and address can be shared. the underscore _ in python is a convention signifying that the information is intended to be internal. 
Python generates a new random SECP256k1 private key, and subsequently derives the corresponding public key.
Following this, the public key is converted into a hexadecimal string which can be shared.
The private key, which is not to be shared is also converted into a hexdecimal string for the wallet owner and is used to create a digital signature.. 
The wallet address is derived from the public key.


In [7]:
class Wallet:

    def __init__(self, signing_key: SigningKey, label: str = "") -> None:
        self.label = label
        self._sk = signing_key
        self._vk = signing_key.get_verifying_key()

    @classmethod
    def create(cls, label: str = "") -> "Wallet":
        return cls(SigningKey.generate(curve=SECP256k1), label=label)

    @property
    def public_key_hex(self) -> str:
        return self._vk.to_string().hex()

    @property
    def private_key_hex(self) -> str:
        return self._sk.to_string().hex()

    @property
    def address(self) -> str:
        return pubkey_to_address(self._vk.to_string())

    def sign(self, message: bytes) -> str:
        return self._sk.sign(message).hex()

    @staticmethod
    def verify(message: bytes, signature_hex: str, public_key_hex: str) -> bool:
        vk = VerifyingKey.from_string(bytes.fromhex(public_key_hex), curve=SECP256k1)
        try:
            return vk.verify(bytes.fromhex(signature_hex), message)
        except BadSignatureError:
            return False

    def sign_transaction(
        self, recipient_address: str, amount: float, timestamp: float
    ) -> str:
        payload = canonical_tx_payload(self.address, recipient_address, amount, timestamp)
        return self.sign(payload)

    def __repr__(self) -> str:
        return f"Wallet(label={self.label!r}, address={self.address!r})"

#static method verifies the signature. verification requires the original message, the signature as well as the wallet owner's public key.
#vk = VerifyingKey.from_string( bytes.fromhex(public_key_hex),curve=SECP256k1) reconstructs the public key from its hexadecimal representation.
Following this, #return vk.verify(bytes.fromhex(signature_hex), message) verifies whether the message was signed using the private key corresponding to the public key. If yes, python returns True and False if the message or signature has been altered.
#try:return vk.verify(...) except BadSignatureError: return False rather than crashing the blockchain program, this function causes an invalid signature to compute a BadSignatureError


Transaction signing:
First, #payload = canonical_tx_payload(self.address,recipient_address,amount,timestamp) creates the canonical transaction bytes discussed earlier. 
Next, #return self.sign(payload) uses the wallet owner's private key to sign these exact bytes. 
#_repr_(self)-> str: controls the output observed when the wallet is printed, purposefully omitting the private key.  


In [8]:
!jupyter nbconvert --to script Wallet_blchA4.ipynb

[NbConvertApp] Converting notebook Wallet_blchA4.ipynb to script
[NbConvertApp] Writing 6269 bytes to Wallet_blchA4.py
